# 第8回 演習: Forth インタプリタの実装

スライド: [第8回 (PDF)](https://3tty0n.github.io/2026-scripting-lab/08-forth/08-forth-beamer.pdf)

**インタプリタ** (プログラムを読んで、その場で実行するプログラム) を作る。
相手の言語は **Forth**。数と命令を空白で区切って並べるだけの、とても小さな言語である。
Forth の命令1つ1つを**ワード**と呼ぶ。

土台は**第7回で作った逆ポーランド電卓** (`WORDS` と `evaluate`)。
そこに「新しいワードを自分で定義する仕組み」を足すと、電卓が言語になる。

```
3 4 + .          →  7     (3 と 4 を積み、足して、表示する)
: SQUARE DUP * ;          (SQUARE という新しいワードを定義する)
5 SQUARE .       →  25
```

**スタック**: 皿を積み重ねるように、最後に置いたものから取り出すリスト。
Python では普通のリストで、`append` で一番上に積み (push)、`pop` で一番上を取り出す (pop)。

## このノートブックの使い方

1. **一番上のセルから順に、全部実行する** (`ランタイム → すべてのセルを実行` でもよい)
2. `TODO` と書かれた行、`pass`、`...` が、自分で書く場所。`TODO` の指示どおりに書き換える
3. 書いたら、そのすぐ下の **確認セル** (先頭が `# [CHECK]`) を実行する。`OK` と出れば正解
4. 確認セルは書き換えない。`NG` や赤いエラーが出たら、表示された行やメッセージを手がかりに直す
5. `NameError` (名前が見つからない) が出たら、上のセルを実行し忘れている。上から実行し直す
6. **課題1・2 は、第7回の課題4・4-b で書いたコードをそのまま貼ってよい** (名前が同じ)

## 全体の地図

インタプリタは次の関数でできている。**★が自分で書く部分**で、他は用意してある。

```
forth_eval(stack, text)          用意済み: ワードを左から1つずつ取り出して実行する
 ├─ read_definition(tokens, i)   ★課題3   ': 名前 本体 ;' を読み取る
 ├─ eval_user(stack, word)       ★課題3-b 自分で定義したワードを実行する
 └─ evaluate(stack, text)        ★課題1   第7回の電卓: WORDS を引いて実行する
                                 ★課題2   WORDS のワードを増やす (. DUP DROP SWAP OVER ROT)
repl(lines)                      ★課題4   1行ずつ実行して結果を返すループ
```

| 課題 | 書くもの | 配点 |
|---|---|---|
| 課題1 | `evaluate`, 四則演算 (第7回の課題4) | 20 |
| 課題2 | `.` とスタック操作ワード (第7回の課題4-b) | 20 |
| 課題3 | `read_definition` | 15 |
| 課題3-b | `eval_user` | 10 |
| 課題4 | `repl` | 15 |
| 発展1 | 比較演算子 | 10 |
| 発展2 | `IF / ELSE / THEN` | 10 |

---
## 課題0: 用意されたものを読む (採点なし)

次のセルを実行する。**この後の課題で埋める「枠」** がここにある。書き換えなくてよい。

- `WORDS`: 組み込みワードの表 (辞書)。鍵がワードの名前、値がそのワードの関数。第7回と同じ。課題1・2 で増やす
- `USER_WORDS`: 自分で定義したワードの表。鍵が名前、値が本体 (ワードのリスト)。課題3-b で使う
- `forth_eval(stack, text)`: ワードを左から1つずつ実行するループ。
  `:` なら定義を読み、自分で定義したワードなら本体を実行し、それ以外は**第7回の `evaluate` に任せる**
- `run(text)`: 空のスタックから実行して、最後のスタックを返す。確認セルはこれを使う
- `trace(text)`: 1語ごとにスタックの様子を表示する。うまく動かないときに使う

```
run('3 4 +')      →  [7]
trace('3 4 +')    →      3  [3]
                         4  [3, 4]
                         +  [7]
```

In [ ]:
WORDS = {}          # 組み込みワード。第7回と同じ。課題1, 2 で増やす
USER_WORDS = {}     # 自作ワード。名前 -> 本体 (ワードのリスト)


def forth_eval(stack, text):
    """ワードを左から1つずつ実行する。text は文字列でもワードのリストでもよい"""
    tokens = text.split() if isinstance(text, str) else text
    i = 0  # 今見ているワードの位置
    while i < len(tokens):
        token = tokens[i]
        key = token.upper()
        # ':' なら定義を読み取り、USER_WORDS に登録する
        if token == ':':
            name, body, i = read_definition(tokens, i)
            USER_WORDS[name] = body
            continue
        # 発展2: IF / ELSE / THEN の枝はここに足す
        # 自分で定義したワードなら本体を実行。それ以外は第7回の evaluate に任せる
        if key in USER_WORDS:
            eval_user(stack, key)
        else:
            evaluate(stack, key)
        i += 1


def run(text):
    """空のスタックから text を実行して、できたスタックを返す"""
    stack = []
    forth_eval(stack, text)
    return stack


def trace(text):
    """1語ごとにスタックを表示する (デバッグ用。定義 : ... ; は含めないこと)"""
    stack = []
    for token in text.split():
        forth_eval(stack, [token])
        print(f'{token:>8}  {stack}')

In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '08'


---
## 課題1: 電卓 — 第7回の課題4 (20点)

第7回の課題4 と同じ。**第7回で書いたコードがあれば、そのまま下のセルに貼ってよい。**

**やること**: 四則演算のワード `- * /` を作り、`evaluate` を書いて、数式を計算できるようにする。

**入出力の例**:

```
run('3 4 +')        →  [7]
run('10 3 -')       →  [7]       (10 - 3)
run('3 4 + 2 *')    →  [14]      ((3 + 4) * 2)
run('20 4 /')       →  [5]
run('3 FOO')        →  ValueError (FOO というワードは無い)
```

**手順** (下のセルの `TODO` と同じ順):

1. `- * /` を、`+` の `add` と同じ形で関数に書き、`WORDS['-'] = sub` のように表に入れる。割り算は `//` (整数の割り算)
2. `evaluate` の中身を書く。空白で区切った1語ずつについて
   - `WORDS` にある語なら、その関数にスタックを渡して呼ぶ (`WORDS[token](stack)`)
   - 数なら、`int(token)` にして積む
   - どちらでもなければ `ValueError` を出す (`raise ValueError(...)`)

**ヒント**: `token.lstrip('-').isdigit()` で、`-5` のような負の数も含めて「数かどうか」を判定できる。

**つまずきやすい点**:

- 引き算・割り算の**順序**。先に `pop` されるのは**右側** (後に積んだ方) の値。`b, a = s.pop(), s.pop()` として `a - b`。
  確認セルの `assert run('10 3 -') == [7]` で止まったら、ここを疑う
- `/` を `/` のまま書くと `5.0` (小数) になる。`//` を使う
- 「知らないワードは ValueError を出すこと」と出たら、3つ目の場合 (どちらでもない) を書き忘れている

### 例題

次のセルを実行して、動きを見る。

`3 4 +` を手で追う。スタックは `append` で積み、`pop` で取る。

In [ ]:
# 例題
s = []
s.append(3)
s.append(4)
b = s.pop()
a = s.pop()
s.append(a + b)
print(s)    # [7]

s = [10, 3]
b, a = s.pop(), s.pop()
print(a - b)    # 7

`+` は書いてあるので、`- * /` を同じ形で足す。
`evaluate` は、第7回と同じく「`WORDS` にあれば呼ぶ・数なら積む・それ以外は `ValueError`」。

In [ ]:
def add(s):
    b, a = s.pop(), s.pop()
    s.append(a + b)


WORDS['+'] = add

# TODO: - * / も、+ と同じ形で関数を書いて WORDS に入れる (/ は整数除算 //)


def evaluate(stack, text):
    for token in text.split():
        # TODO: WORDS にあれば呼ぶ。なければ整数として積む。どちらでもなければ ValueError
        pass


print(run('3 4 +'))    # [7]
print(run('10 3 -'))   # [7]
trace('3 4 + 2 *')

In [ ]:
# [CHECK] q1
need('WORDS', kind=dict)
need('evaluate', arity=2)
need('forth_eval', arity=2)
need('run', arity=1)

assert run('3 4 +') == [7]
assert run('10 3 -') == [7]        # 順序に注意
assert run('3 4 + 2 *') == [14]
assert run('2 3 4 * +') == [14]
assert run('20 4 /') == [5]
assert run('1 2 3') == [1, 2, 3]
assert run('-5 3 +') == [-2]
assert run('') == []

# 性質で確かめる
holds('数を並べたらそのまま積まれる',
      lambda a, b: run(f'{a} {b}') == [a, b], ints(2, lo=0, hi=99))
holds('足してから引くと元の数',
      lambda a, b: run(f'{a} {b} + {b} -') == [a], ints(2, lo=0, hi=99))
try:
    run('3 FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')

# WORDS に後から足したワードも、evaluate を変えずに使えること
def _neg(s):
    s.append(-s.pop())
WORDS['NEG'] = _neg
assert run('5 NEG') == [-5], '実行中にワードを増やせること'
del WORDS['NEG']
print('OK')

---
## 課題2: `.` とスタック操作ワード — 第7回の課題4-b (20点)

**やること**: 下の表のワードを、関数に書いて `WORDS` に入れる。**`evaluate` は変更しない**。
第7回の課題4-b で書いた分があれば貼ってよい。`.` (ドット) だけが新しい。

| ワード | 動作 | スタックの変化 |
|---|---|---|
| `.` | 一番上を**表示して**取り除く | (a → ) |
| `DUP` | 一番上を複製 | (a → a a) |
| `DROP` | 一番上を捨てる | (a → ) |
| `SWAP` | 上2つを入れ替え | (a b → b a) |
| `OVER` | 2番目を上にコピー | (a b → a b a) |
| `ROT` | 3番目を一番上へ | (a b c → b c a) |

`(a b → b a)` は「左が実行前、右が実行後」のスタック。右端が一番上。

**入出力の例**:

```
run('5 DUP')        →  [5, 5]
run('1 2 SWAP')     →  [2, 1]
run('1 2 3 ROT')    →  [2, 3, 1]
run('42 .')         →  42 と表示され、スタックは []
```

**手順** (下のセルの `TODO` と同じ順): `dot`, `dup`, `drop`, `swap`, `over`, `rot` の中身を1つずつ書く。
表に入れる行 (`WORDS['DUP'] = dup` など) は書いてある。

**ヒント**:

- 一番上は `stack[-1]`、2番目は `stack[-2]` で、取り出さずに覗ける
- `DUP` は「一番上を覗いて、もう1つ積む」。`.` は「取り出して `print`」
- 迷ったら、先に紙にスタックの図を書いてからコードにする

**つまずきやすい点**:

- `stack.pop()` は値を**取り出す** (スタックから消える)。`stack[-1]` は**覗くだけ**。`DUP` や `OVER` で `pop` すると数が減る
- 名前は大文字で入れる。`forth_eval` が大文字にそろえてから渡すので、`dup` と小文字で書いても動く
- 「ドットは取り除くこと」「ドットは出力すること」と出たら、`.` の `pop` か `print` を忘れている

### 例題

次のセルを実行して、動きを見る。

ワードを1つ作って、`WORDS` に入れる形。

In [ ]:
# 例題
def neg(stack):
    stack.append(-stack.pop())


WORDS['NEG'] = neg
print(run('5 NEG'))    # [-5]
del WORDS['NEG']


In [ ]:
def dot(stack):
    # TODO: 一番上を取り出して print する
    pass


WORDS['.'] = dot


def dup(stack):
    # TODO: 一番上 stack[-1] を、もう1つ積む
    pass


WORDS['DUP'] = dup


def drop(stack):
    # TODO: 一番上を取り出して捨てる
    pass


WORDS['DROP'] = drop


def swap(stack):
    # TODO: 上2つを取り出し、順番を入れ替えて積み直す
    pass


WORDS['SWAP'] = swap


def over(stack):
    # TODO: 2番目 stack[-2] を、一番上に積む
    pass


WORDS['OVER'] = over


def rot(stack):
    # TODO: 上3つを取り出し、3番目だったものが一番上になるよう積み直す
    pass


WORDS['ROT'] = rot


print(run('1 2 SWAP'))    # [2, 1]
trace('1 2 3 ROT')


In [ ]:
# [CHECK] q2
need('WORDS', kind=dict)
need('forth_eval', arity=2)

import io, contextlib

assert run('1 2 SWAP') == [2, 1]
assert run('1 2 swap') == [2, 1]     # 小文字でも動くこと
assert run('5 DUP') == [5, 5]
assert run('1 2 DROP') == [1]
assert run('1 2 OVER') == [1, 2, 1]
assert run('1 2 3 ROT') == [2, 3, 1]
assert run('5 dup *') == [25]
assert run('3 4 + 2 *') == [14]      # 課題1 が壊れていないこと
assert run('10 3 -') == [7]

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    s = run('42 .')
assert s == [], 'ドットは取り除くこと'
assert '42' in buf.getvalue(), 'ドットは出力すること'

try:
    run('FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')

---
## 課題3: ワードの定義を読む (15点)

**やること**: `: 名前 本体 ;` という定義を読み取る `read_definition(tokens, i)` を書く。
読み取るだけで、実行はしない (登録は用意済みの `forth_eval` がする)。

- `tokens`: 入力を空白で区切ったリスト
- `i`: `:` のある位置 (`tokens[i]` が `':'`)
- 返すもの: `(名前を大文字にしたもの, 本体のワードのリスト, ';' の次の位置)` の3つ組

**入出力の例**:

```
tokens = [':', 'SQUARE', 'DUP', '*', ';', '5', 'SQUARE']
位置      0      1        2     3    4    5     6

read_definition(tokens, 0)  →  ('SQUARE', ['DUP', '*'], 5)
```

`5` は `;` (位置4) の次の位置。`forth_eval` はそこから続きを読む。

**手順** (下のセルの `TODO` と同じ順):

1. `j = tokens.index(';', i)` で、位置 `i` より後ろで最初の `;` の位置 `j` を探す
2. `:` のすぐ後が `;` (名前が無い。`j` が `i + 1`) なら `ValueError` を出す
3. `(tokens[i + 1].upper(), tokens[i + 2:j], j + 1)` を返す

**ヒント**: `tokens.index(';', i)` は、`;` が無いと自分で `ValueError` を出す。
`;` が無い場合は、何も書かなくても条件を満たす。

**つまずきやすい点**:

- 名前を大文字にし忘れる。確認セルに「名前は大文字にすること」と出る
- 返す位置を `j` (`;` の位置) にしてしまう。`;` の**次** `j + 1` を返す
- `tokens.index(';')` と `i` を付け忘れると、2つ目の定義で1つ目の `;` を拾ってしまう (「2つ目の定義」で止まる)

### 例題

次のセルを実行して、動きを見る。

`;` の位置を探して、スライス (リストの一部を `[始め:終わり]` で切り出すこと) で取り出す。

In [ ]:
# 例題
tokens = ': SQUARE DUP * ; 5 SQUARE'.split()
j = tokens.index(';', 0)
print(j)               # 4
print(tokens[1])       # SQUARE
print(tokens[2:j])     # ['DUP', '*']
print(j + 1)           # 5

In [ ]:
def read_definition(tokens, i):
    # TODO: j = tokens.index(';', i) で ; の位置を探す
    # TODO: 名前が無い (j が i + 1) なら ValueError を出す
    # TODO: (名前を大文字にしたもの, tokens[i + 2:j], j + 1) を返す
    pass


print(read_definition(': SQUARE DUP * ; 5 SQUARE'.split(), 0))   # ('SQUARE', ['DUP', '*'], 5)

In [ ]:
# [CHECK] q3
need('read_definition', arity=2)

assert read_definition(': SQUARE DUP * ; 5'.split(), 0) == ('SQUARE', ['DUP', '*'], 5)
assert read_definition('1 : inc 1 + ; inc'.split(), 1) == ('INC', ['1', '+'], 6), '名前は大文字にすること'
assert read_definition(': NOP ;'.split(), 0) == ('NOP', [], 3), '本体が空でもよい'
assert read_definition(': A B ; : C D ;'.split(), 4) == ('C', ['D'], 8), '2つ目の定義'

for bad in (': FOO DUP', ':', ': ;'):
    try:
        read_definition(bad.split(), 0)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{bad!r} は ValueError にすること')
print('OK')

---
## 課題3-b: 自分で定義したワードを実行する (10点)

用意済みの `forth_eval` は、`:` を見つけると `read_definition` で読み取り、
`USER_WORDS[名前] = 本体` に入れる。

**やること**: `SQUARE` のような自分で定義したワードが呼ばれたときに、
その本体を実行する `eval_user(stack, word)` を書く。

**入出力の例**:

```
run(': SQUARE DUP * ;  5 SQUARE')                                →  [25]
run(': DOUBLE DUP + ; : QUADRUPLE DOUBLE DOUBLE ;  5 QUADRUPLE')   →  [20]
```

`USER_WORDS` の中身は、例えば `{'SQUARE': ['DUP', '*']}` になっている。

**手順** (下のセルの `TODO`): 本体 `USER_WORDS[word]` を `forth_eval(stack, 本体)` に渡す。**1行で書ける**。

**ヒント**: `forth_eval` は文字列だけでなく、**ワードのリストも受け取れる**。
`QUADRUPLE` の本体の中の `DOUBLE` も、`forth_eval` がまた `eval_user` を呼ぶので、自然に動く。

**つまずきやすい点**:

- 本体を `WORDS` に入れてしまう。確認セルに「ユーザ定義を WORDS に混ぜないこと」と出る。`eval_user` は表に入れず、実行するだけ
- `evaluate(stack, ...)` を呼ぶと、`DOUBLE` のような自分で定義したワードを知らないので止まる。`forth_eval` を呼ぶ

### 例題

次のセルを実行して、動きを見る。

リストを渡しても `forth_eval` は動く。

In [ ]:
# 例題
s = [5]
forth_eval(s, ['DUP', '*'])
print(s)    # [25]

In [ ]:
def eval_user(stack, word):
    # TODO: 本体 USER_WORDS[word] を forth_eval に渡して実行する
    pass


USER_WORDS.clear()
print(run(': SQUARE DUP * ;  5 SQUARE'))                                  # [25]
print(run(': DOUBLE DUP + ; : QUADRUPLE DOUBLE DOUBLE ;  5 QUADRUPLE'))   # [20]

In [ ]:
# [CHECK] q3b
need('USER_WORDS', kind=dict)
need('eval_user', arity=2)
need('forth_eval', arity=2)

USER_WORDS.clear(); assert run(': SQUARE DUP * ; 5 SQUARE') == [25]
USER_WORDS.clear(); assert run(': DOUBLE DUP + ; : QUAD DOUBLE DOUBLE ; 5 QUAD') == [20]
USER_WORDS.clear(); assert run(': INC 1 + ; 10 INC INC INC') == [13]
USER_WORDS.clear(); assert run('1 : ADD2 2 + ; ADD2 ADD2') == [5]
USER_WORDS.clear(); assert run(': SQ DUP * ; 3 SQ SQ') == [81]

# 定義しただけでは何もスタックに積まれない
USER_WORDS.clear(); assert run(': NOP ;') == []

# 大文字小文字を区別しないこと
USER_WORDS.clear(); assert run(': SQUARE DUP * ; 4 square') == [16]

# 組み込みが壊れていないこと
USER_WORDS.clear()
assert run('3 4 + 2 *') == [14]
assert run('1 2 3 ROT') == [2, 3, 1]
# ユーザ定義は USER_WORDS に入れ、組み込みの表は汚さないこと
USER_WORDS.clear()
_before = set(WORDS)
run(': SQUARE DUP * ;')
assert set(WORDS) == _before, 'ユーザ定義を WORDS に混ぜないこと'
assert len(USER_WORDS) == 1, f'USER_WORDS が {len(USER_WORDS)} 件'
note('ユーザ定義ワードは組み込みと別の表に入っている')

print('OK')

---
## 課題4: 対話ループ (REPL) (15点)

**REPL** は、1行読んで、実行して、結果を表示する、をくり返す仕組み (Python の `>>>` と同じ)。
採点できるように、`input()` で読むかわりに、**入力行のリストを引数で受け取る**形で書く。

**やること**: `repl(lines)` を書く。`lines` は入力行のリスト。
返すのは、**各行に対する出力の文字列のリスト**。

| 入力行 | 返すもの |
|---|---|
| 普通の行 | 実行後のスタックを `str()` で文字列にしたもの (例: `'[7]'`) |
| `bye` | そこで終了。それ以降の行は処理しない |
| エラーが起きた行 | `'error: ' + エラーメッセージ`。**ループは続ける** |

**入出力の例**:

```
repl(['3 4 +', '2 *'])          →  ['[7]', '[14]']   (2行目は 1行目のスタック [7] の続き)
repl(['1 2 +', 'bye', '999'])   →  ['[3]']           (bye の後は無視)
repl(['1 2 +', 'FOO', '10 +'])  →  ['[3]', 'error: ...', '[13]']
```

**手順** (下のセルの `TODO` と同じ順):

1. 行が `bye` なら、`break` でループを抜ける (前後の空白や大文字は `line.strip().lower()` でそろえる)
2. `try:` の中で `forth_eval(stack, line)` を実行し、`str(stack)` を `outputs` に足す
3. `except Exception as e:` でエラーを捕まえ、`'error: ' + str(e)` を `outputs` に足す

**ヒント**: 下の例題が、ほぼそのままの形。`bye` の判定を足せばよい。

**つまずきやすい点**:

- `stack = []` をループの**中**に書くと、行ごとにスタックが空に戻る。「エラーのあともスタックが保たれること」と出る。`stack` はループの外 (書いてある所) のまま
- `bye` を `try` の中で判定すると、`bye` 自体が知らないワードとしてエラーになる。`try` の前で判定する

### 例題

次のセルを実行して、動きを見る。

1行ずつ、失敗しても続ける形。

In [ ]:
# 例題
outputs = []
stack = []
for line in ['3 4 +', 'FOO']:
    try:
        forth_eval(stack, line)
        outputs.append(str(stack))
    except Exception as e:
        outputs.append('error: ' + str(e))
print(outputs)    # ['[7]', 'error: ...']

`stack` をループの外で作っているので、値は行をまたいで残る。

In [ ]:
def repl(lines):
    stack = []
    outputs = []
    for line in lines:
        # TODO: bye なら break する
        # TODO: try の中で forth_eval(stack, line) を実行し、str(stack) を outputs に足す
        # TODO: except でエラーを捕まえ、'error: ' + str(e) を outputs に足す
        pass
    return outputs


USER_WORDS.clear()
for out in repl(['3 4 +', '2 *', 'FOO', ': SQ DUP * ;', 'SQ', 'bye', '999']):
    print(out)

In [ ]:
# 対話用 (採点しない)。実行すると入力待ちになり、bye で終わる
def repl_interactive():
    stack = []
    while True:
        line = input('> ')
        if line.strip().lower() == 'bye':
            break
        try:
            forth_eval(stack, line)
            print(stack)
        except Exception as e:
            print('error:', e)


# repl_interactive()   ← コメントを外すと対話できる

In [ ]:
# [CHECK] q4
need('repl', arity=1)

USER_WORDS.clear()
out = repl(['3 4 +', '2 *'])
assert out == ['[7]', '[14]'], out

# bye で止まり、それ以降は処理しない
USER_WORDS.clear()
out = repl(['1 2 +', 'bye', '999'])
assert out == ['[3]'], out

# エラーでもループは続く
USER_WORDS.clear()
out = repl(['1 2 +', 'FOO', '10 +'])
assert len(out) == 3, out
assert out[0] == '[3]'
assert out[1].startswith('error:'), out[1]
assert out[2] == '[13]', 'エラーのあともスタックが保たれること'

# ユーザ定義ワードも行をまたいで残る
USER_WORDS.clear()
out = repl([': SQ DUP * ;', '5 SQ'])
assert out == ['[]', '[25]'], out

assert repl([]) == []
assert repl(['bye']) == []
print('OK')


---
# 発展課題

余力があれば。基本課題 (課題1〜4) だけで 80 点。

## 発展1: 比較演算子 (10点)

**やること**: 比較のワード `<` `>` `=` を `WORDS` に入れる。`<` は例題に書いてあるので、`>` と `=` を書く。

Forth では、**真 (正しい) を `-1`、偽 (正しくない) を `0`** で表す。

**入出力の例**:

```
run('3 4 <')    →  [-1]    (3 < 4 は正しい)
run('3 4 >')    →  [0]
run('4 4 =')    →  [-1]
```

**手順**: `less` をまねて `greater` と `equal` を書き、`WORDS['>']`, `WORDS['=']` に入れる。

**つまずきやすい点**: Python の `True` をそのまま積むと `1` になる。`-1 if a > b else 0` のように `-1` / `0` に直す。

### 例題

次のセルを実行して、動きを見る。

比較の結果を Forth の真偽値 (`-1` / `0`) に直す。

In [ ]:
# 例題
def less(stack):
    b, a = stack.pop(), stack.pop()
    stack.append(-1 if a < b else 0)


WORDS['<'] = less


`>` と `=` も同じ形。

In [ ]:
# TODO: > と = も、< と同じ形で関数を書いて WORDS に入れる


In [ ]:
# [CHECK] b1
USER_WORDS.clear()
assert run('3 4 <') == [-1]
assert run('4 3 <') == [0]
assert run('3 4 >') == [0]
assert run('4 3 >') == [-1]
assert run('4 4 =') == [-1]
assert run('3 4 =') == [0]
assert run('4 4 <') == [0] and run('4 4 >') == [0]
assert run('1 2 + 3 =') == [-1]
print('OK')


## 発展2: 条件分岐 (10点)

**やること**: `IF ... ELSE ... THEN` を使えるようにする。
`IF` の直前にスタックの一番上にある値が 0 以外なら `IF` と `ELSE` の間を、
0 なら `ELSE` と `THEN` の間を実行する (`ELSE` は省略できる)。

**入出力の例**:

```
run('1 IF 10 ELSE 20 THEN')    →  [10]
run('0 IF 10 ELSE 20 THEN')    →  [20]
: ABS DUP 0 < IF 0 SWAP - THEN ;
-5 ABS .                       →  5
```

**どこを直すか**: 課題0 のセルの `forth_eval`。`# 発展2` の目印の行に、`IF` の場合を足す
(`:` の場合と同じく、`i` を進めて `continue`)。

**手順**:

1. `key == 'IF'` なら、条件 `stack.pop()` を取り出す
2. 対応する `ELSE` と `THEN` の位置を探す (例題の `find_then` の考え方)
3. 条件に応じて、実行する側のワードのリストを切り出し、`forth_eval(stack, 切り出した部分)` に渡す
4. `i` を `THEN` の次に進めて `continue`

**ヒント**: `IF` の中にまた `IF` があることがある (入れ子)。深さ `depth` を数え、
`depth == 1` の `ELSE` だけを自分のものとして拾う。

### 例題

次のセルを実行して、動きを見る。

入れ子を数えながら、対応する `THEN` を探す。

In [ ]:
# 例題
def find_then(tokens, i):
    """tokens[i] == 'IF' のとき、対応する THEN の位置を返す"""
    depth = 1
    j = i + 1
    while j < len(tokens):
        if tokens[j].upper() == 'IF':
            depth += 1
        elif tokens[j].upper() == 'THEN':
            depth -= 1
            if depth == 0:
                return j
        j += 1
    raise ValueError('THEN がない')

`ELSE` も同じ要領で探す。見つけた位置を使ってリストを切り出し、`forth_eval` に渡す。

In [ ]:
# TODO: 課題0 のセルの forth_eval に IF / ELSE / THEN を足す (このセルは空のままでよい)

In [ ]:
# [CHECK] b2
USER_WORDS.clear()
assert run('1 IF 42 THEN') == [42]
assert run('0 IF 42 THEN') == []
assert run('1 IF 10 ELSE 20 THEN') == [10]
assert run('0 IF 10 ELSE 20 THEN') == [20]
assert run('-1 IF 7 THEN') == [7], '0 以外はすべて真'

USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; -5 ABS') == [5]
USER_WORDS.clear()
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; 5 ABS') == [5]

# 入れ子
USER_WORDS.clear()
assert run('1 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [7]
assert run('1 IF 0 IF 7 ELSE 8 THEN ELSE 9 THEN') == [8]
assert run('0 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [9]

# 条件分岐の外側は普通に続く
assert run('1 IF 10 THEN 5 +') == [15]
print('OK')


---
## 自己確認

`score()` で全部の確認をまとめて実行する。何度やってもよい。
`NG` の行に、次に何をすればいいかが出る。

In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` は下書きの確認用で、提出にはならない。

1. 一番上から全部実行し、出力を残したまま保存する
2. `ファイル → .ipynb をダウンロード`
3. 締切までに kibaco へ出す

確認セルが全部 `OK` なら提出してよい。確認セルを消しても、教員の採点は別の検査で行う。